# T10b — 반복 루프 대조군 + DTH 시드 0

1. **대조군:** 적응 없이 반복만 막는 디코딩(`no_repeat_ngram_size=5`)을 b0와 KEJ 어댑터에 똑같이 건다.
   b0가 이것만으로 b1에 근접하면, 어댑터의 기여는 대부분 "반복을 멈추게 한 것"이다.
2. **DTH 시드 0:** 학습 54쌍(5.7분) → 02-04 파일 전체 평가. 화자 2명째.

## 드라이브 `MyDrive/t10_highcer/` 에 덮어쓸 것

- `t10_pairs_manifest.json` (DTH 추가됨 — **기존 파일 먼저 삭제**)
- `eval_longform.py` (`--no-repeat` 추가됨 — **기존 파일 먼저 삭제**)
- `segments/` 폴더: 로컬에 136개. 드라이브에 없는 DTH 파일 109개를 추가로 올린다

**런타임 T4.** 약 20분.

## 1. 설치

In [ ]:
!pip -q install peft rapidfuzz ; pip -q uninstall -y torchao ; python -c "import torch, importlib.util as u; print('GPU', torch.cuda.is_available(), 'torchao 없음' if u.find_spec('torchao') is None else 'TORCHAO 남아있음 - 런타임 재시작')"


## 2. 드라이브

In [ ]:
from google.colab import drive; drive.mount('/content/drive')


## 3. 업로드 확인 — `segments 136`, `DTH pairs 109`가 나와야 한다

In [ ]:
!cd "/content/drive/MyDrive/t10_highcer" && ls segments | wc -l && python -c "import json;m=json.load(open('t10_pairs_manifest.json'));print('DTH pairs',sum(x['speaker']=='DTH' for x in m['items']))" && grep -c no-repeat eval_longform.py


## 4. 대조군: b0 + 반복 방지 (KEJ)

In [ ]:
!cd "/content/drive/MyDrive/t10_highcer" && python eval_longform.py --id ID-01-13-N-KEJ-02-04-F-36-KK --wavdir . --no-repeat 5


## 5. 대조군: KEJ 어댑터 + 반복 방지

드라이브의 `results/KEJ_nall_s0/adapter` 는 9/22 재실행(run 2) 어댑터다.

In [ ]:
!cd "/content/drive/MyDrive/t10_highcer" && python eval_longform.py --id ID-01-13-N-KEJ-02-04-F-36-KK --wavdir . --no-repeat 5 --adapter results/KEJ_nall_s0/adapter


## 6. DTH 시드 0 학습

`selected epoch`와 `RUNAWAY` 줄을 확인. **출력 전체를 복사해 둘 것.**

In [ ]:
!cd "/content/drive/MyDrive/t10_highcer" && python b1_train.py --speaker DTH --seed 0 --manifest t10_pairs_manifest.json --segdir segments --out results


## 7. DTH b1 평가

In [ ]:
!cd "/content/drive/MyDrive/t10_highcer" && python eval_longform.py --id ID-01-13-N-DTH-02-04-M-85-KK --wavdir . --adapter results/DTH_nall_s0/adapter


## 끝나면

`t10_highcer/results/` 를 통째로 받아 로컬 `experiments/t10_highcer/results/` 에 덮어쓴다. 이전 실행은 `runs/`에 따로 보관돼 있어 덮어써도 된다.

| 증상 | 조치 |
| --- | --- |
| 3번에서 `segments` 가 136이 아님 | DTH 구간 파일 업로드가 덜 됐다 |
| 4번 `unexpected keyword ... no_repeat` 류 오류 | 알려줄 것. 주말 대조군 설정을 바꿔야 한다 |
| `speaker DTH not in manifest` | 매니페스트가 옛 파일이다 (`(1)` 붙은 파일 확인) |